Репозиторий уже скачан; зависимости установлены, включён GPU. Запускай ячейки по порядку. Результат — `predictions.json` в порядке test. Клонирование репозитория добавишь отдельно.

In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter

import torch
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from huggingface_hub import snapshot_download
from tokenizers import Tokenizer

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from src.model import Seq2SeqTransformer, Seq2SeqTransformerConfig
from src.code_preprocessing import restore_code

REPO_ID = "YaItco/NLP-2-transformer"
DEVICE = "cuda"
BATCH_SIZE = 64
NUM_BEAMS = 1
MAX_NEW_TOKENS = 128
OUTPUT = "predictions.json"

In [ ]:
folder = Path(snapshot_download(REPO_ID, allow_patterns=["model.pt", "config.json", "tokenizer.json"]))
config = json.loads((folder / "config.json").read_text())
model = Seq2SeqTransformer(Seq2SeqTransformerConfig(**config["model"]))
model.load_state_dict(torch.load(folder / "model.pt", map_location="cpu", weights_only=True))
model = model.to(DEVICE).eval()
tokenizer = Tokenizer.from_file(str(folder / "tokenizer.json"))
pad_id = config["model"]["pad_id"]

In [ ]:
test = load_dataset("SlavaYus/tinysms_qa_12m", split="test", streaming=True)
questions = [row["question"] for row in test]

In [ ]:
torch.cuda.synchronize()
start = perf_counter()
encoded = tokenizer.encode_batch(questions, add_special_tokens=False)
order = sorted(range(len(questions)), key=lambda i: len(encoded[i].ids))
predictions = [None] * len(questions)

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    for offset in range(0, len(order), BATCH_SIZE):
        indices = order[offset:offset + BATCH_SIZE]
        rows = [torch.tensor(encoded[i].ids, dtype=torch.long) for i in indices]
        input_ids = pad_sequence(rows, batch_first=True, padding_value=pad_id).to(DEVICE)
        generated = model.generate(
            input_ids, attention_mask=input_ids.ne(pad_id),
            max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS, use_cache=True,
        )
        codes = tokenizer.decode_batch(generated.cpu().tolist(), skip_special_tokens=True)
        for i, code in zip(indices, codes):
            predictions[i] = restore_code(code) if config["code_format"] == "solve_body_result" else code

torch.cuda.synchronize()
print(f"{len(predictions)} examples, {perf_counter() - start:.1f} s (tokenization + generation)")

In [ ]:
Path(OUTPUT).write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8");